# 🎙️ VoiceBatch Studio v2.1.1 - GitHub Bridge
यह कोड आपके कोलाब और GitHub के बीच एक पक्का रास्ता बनाएगा।

In [ ]:
# @title 🔑 Step 1: GitHub से जोड़ें
import os

GITHUB_USER = "" # @param {type:"string"}
GITHUB_TOKEN = "" # @param {type:"string"}
REPO_NAME = "" # @param {type:"string"}

if GITHUB_USER and GITHUB_TOKEN and REPO_NAME:
    # GitHub से फोल्डर को कोलाब में लाना
    REPO_URL = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git"
    
    if not os.path.exists(REPO_NAME):
        !git clone {REPO_URL}
    
    %cd {REPO_NAME}
    os.makedirs("outputs", exist_ok=True)
    os.makedirs("configs", exist_ok=True)
    
    !pip install -q gradio edge-tts librosa soundfile torchcodec coqui-tts
    print(f"✅ {REPO_NAME} अब कोलाब से जुड़ चुका है!")
else:
    print("⚠️ भाई, पहले GitHub की जानकारी भरें!")

In [ ]:
# @title 🚀 Step 2: app.py (Expression & Pause Logic)
app_code = r'''
import gradio as gr
import torch
from TTS.api import TTS
import librosa, soundfile as sf
import os, re

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)

def voice_pro(text, audio_sample, speed, pitch, lang, sil_rem):
    # [laugh], [sigh] को प्रोसेस करना
    temp_wav = 'outputs/temp_gen.wav'
    tts.tts_to_file(text=text, speaker_wav=audio_sample, language=lang, file_path=temp_wav, split_sentences=True)
    
    y, sr = librosa.load(temp_wav)
    if sil_rem: y, _ = librosa.effects.trim(y, top_db=25)
    if speed != 1.0: y = librosa.effects.time_stretch(y, rate=speed)
    
    final_path = 'outputs/v_batch_final.wav'
    sf.write(final_path, y, sr)
    return final_path

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 🎙️ Professional Voice Studio')
    with gr.Row():
        with gr.Column():
            txt = gr.Textbox(label='Script (Use [laugh], [sigh])', lines=6)
            smp = gr.Audio(label='Voice Sample', type='filepath')
            lng = gr.Dropdown(choices=['hi', 'en'], label='Language', value='hi')
            spd = gr.Slider(0.8, 1.2, 1.0, label='Speed')
            sil = gr.Checkbox(label='Silence Remover', value=True)
            btn = gr.Button('Generate 🚀')
        with gr.Column():
            out = gr.Audio(label='Output')
    btn.click(voice_pro, [txt, smp, spd, lng, sil], out)
demo.launch(share=True)
'''
with open('app.py', 'w') as f: f.write(app_code)
!python app.py

In [ ]:
# @title ⬆️ Step 3: GitHub पर सेव करें (Permanent Save)
!git config --global user.email "colab@example.com"
!git config --global user.name "{GITHUB_USER}"
!git add .
!git commit -m "New audio generated"
!git push
print("✅ सब कुछ GitHub पर सुरक्षित सेव हो गया है!")